In [1]:

import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

file = os.listdir(path)[0]
full_file = path  + '/' + file
DATASET_PATH = path  + '/' + file

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.


In [9]:
import os
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

# ----------------------------------------------------------------------------
# 1. Configuration & Device Setup
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# DATASET_PATH = "IMDB Dataset.csv"
MODEL_NAME = "distilbert-base-uncased"  # Much smaller and faster than BERT
MAX_LENGTH = 256
BATCH_SIZE = 128
EPOCHS = 5

# ----------------------------------------------------------------------------
# 2. Load and Preprocess Data
# ----------------------------------------------------------------------------
df = pd.read_csv(DATASET_PATH)
df = df.iloc[:10000]
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', text)
    return text.strip()

df['review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train size: {len(train_texts)} | Validation size: {len(val_texts)}")

# ----------------------------------------------------------------------------
# 3. Tokenization
# ----------------------------------------------------------------------------
print(f"Initializing {MODEL_NAME} Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizing datasets...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LENGTH)

# ----------------------------------------------------------------------------
# 4. PyTorch Dataset Creation
# ----------------------------------------------------------------------------
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)

# ----------------------------------------------------------------------------
# 5. Model Initialization & Metrics
# ----------------------------------------------------------------------------
print(f"Loading pre-trained {MODEL_NAME} model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# ----------------------------------------------------------------------------
# 6. Training Configuration & Execution
# ----------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./distilbert-mac-classification",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.1,

    fp16=True,
    bf16=False,
    use_cpu=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting DistilBERT training...")
trainer.train()

# ----------------------------------------------------------------------------
# 7. Final Evaluation & Save
# ----------------------------------------------------------------------------
print("\nRunning final validation evaluation...")
eval_results = trainer.evaluate()
print(f"Final Evaluation Results: {eval_results}")

print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_distilbert_imdb")
tokenizer.save_pretrained("./fine_tuned_distilbert_imdb")
print("Model saved successfully to './fine_tuned_distilbert_imdb'!")

# ----------------------------------------------------------------------------
# 8. Inference / Prediction Example
# ----------------------------------------------------------------------------
def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()

    return "Positive" if prediction == 1 else "Negative"

print("\n--- Testing Inference ---")
sample_review = "This movie was absolutely fantastic! The acting was superb and the plot kept me on the edge of my seat."
print(f"Review: '{sample_review}'")
print(f"Predicted Sentiment: {predict_sentiment(sample_review)}")

Using device: cuda
Train size: 8000 | Validation size: 2000
Initializing distilbert-base-uncased Tokenizer...
Tokenizing datasets...
Loading pre-trained distilbert-base-uncased model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting DistilBERT training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.590718,0.276060,0.893500,0.895434,0.884578,0.906561
2,0.278926,0.235504,0.896500,0.898080,0.889756,0.906561
3,0.216624,0.252014,0.893500,0.895945,0.880884,0.911531
4,0.146671,0.256200,0.894000,0.894632,0.894632,0.894632
5,0.120147,0.264359,0.893000,0.893214,0.896794,0.889662


...


Running final validation evaluation...


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.120147,0.235504,5,0.896500,0.898080,0.889756,0.906561


Final Evaluation Results: {'eval_loss': 0.2355038821697235, 'eval_accuracy': 0.8965, 'eval_f1': 0.8980797636632201, 'eval_precision': 0.8897560975609756, 'eval_recall': 0.9065606361829026}
Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully to './fine_tuned_distilbert_imdb'!

--- Testing Inference ---
Review: 'This movie was absolutely fantastic! The acting was superb and the plot kept me on the edge of my seat.'
Predicted Sentiment: Positive


In [6]:

import os
import re
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

# ----------------------------------------------------------------------------
# 1. Configuration & Device Setup
# ----------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# DATASET_PATH = "IMDB Dataset.csv"
MODEL_NAME = "roberta-base"  # Much smaller and faster than BERT
MAX_LENGTH = 256
BATCH_SIZE = 64
EPOCHS = 3

# ----------------------------------------------------------------------------
# 2. Load and Preprocess Data
# ----------------------------------------------------------------------------
df = pd.read_csv(DATASET_PATH)
df = df.iloc[:5000]
def clean_text(text):
    text = re.sub(r'<br\s*/?>', ' ', text)
    return text.strip()

df['review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train size: {len(train_texts)} | Validation size: {len(val_texts)}")

# ----------------------------------------------------------------------------
# 3. Tokenization
# ----------------------------------------------------------------------------
print(f"Initializing {MODEL_NAME} Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizing datasets...")
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=MAX_LENGTH)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=MAX_LENGTH)

# ----------------------------------------------------------------------------
# 4. PyTorch Dataset Creation
# ----------------------------------------------------------------------------
class IMDBDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDBDataset(train_encodings, train_labels)
val_dataset = IMDBDataset(val_encodings, val_labels)

# ----------------------------------------------------------------------------
# 5. Model Initialization & Metrics
# ----------------------------------------------------------------------------
print(f"Loading pre-trained {MODEL_NAME} model...")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# ----------------------------------------------------------------------------
# 6. Training Configuration & Execution
# ----------------------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./distilbert-mac-classification",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    fp16=True,
    bf16=False,
    use_cpu=False,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting DistilBERT training...")
trainer.train()

# ----------------------------------------------------------------------------
# 7. Final Evaluation & Save
# ----------------------------------------------------------------------------
print("\nRunning final validation evaluation...")
eval_results = trainer.evaluate()
print(f"Final Evaluation Results: {eval_results}")

print("Saving fine-tuned model...")
model.save_pretrained("./fine_tuned_distilbert_imdb")
tokenizer.save_pretrained("./fine_tuned_distilbert_imdb")
print("Model saved successfully to './fine_tuned_distilbert_imdb'!")

# ----------------------------------------------------------------------------
# 8. Inference / Prediction Example
# ----------------------------------------------------------------------------
def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(text, padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        prediction = torch.argmax(probs, dim=-1).item()

    return "Positive" if prediction == 1 else "Negative"

print("\n--- Testing Inference ---")
sample_review = "This movie was absolutely fantastic! The acting was superb and the plot kept me on the edge of my seat."
print(f"Review: '{sample_review}'")
print(f"Predicted Sentiment: {predict_sentiment(sample_review)}")

Using device: cuda
Train size: 4000 | Validation size: 1000
Initializing roberta-base Tokenizer...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...


Running final validation evaluation...


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.149319,0.172736,3,0.937000,0.935649,0.944330,0.927126


Final Evaluation Results: {'eval_loss': 0.1727358102798462, 'eval_accuracy': 0.937, 'eval_f1': 0.9356486210418795, 'eval_precision': 0.9443298969072165, 'eval_recall': 0.9271255060728745}
Saving fine-tuned model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully to './fine_tuned_distilbert_imdb'!

--- Testing Inference ---
Review: 'This movie was absolutely fantastic! The acting was superb and the plot kept me on the edge of my seat.'
Predicted Sentiment: Positive
